#Tests

## ¿Funciona? y ¿Qué tan rápido?

*   **Corrección:** ¿el programa produce lo que debería para cada entrada válida? La herramienta son las **pruebas**.
*   **Rendimiento:** ¿cuánto tiempo (o recursos) consume y cómo cambia al crecer la entrada? La herramienta son las **mediciones**.

Son preguntas independientes: se puede ser rápido y equivocarse, o acertar y ser lentísimo. Por eso se trabajan en orden: **primero se asegura la corrección y solo después tiene sentido medir velocidad**. Comparar el tiempo de un algoritmo que da resultados erróneos no aporta nada.

---

## Testing:

El **testing** es el proceso de ejecutar el código con entradas elegidas a propósito y comparar lo que devuelve con lo que **debería** devolver. Toda prueba tiene tres ingredientes:

1.  Una **entrada** concreta.
2.  Un **resultado esperado**, decidido *antes* de ejecutar.
3.  Una **comparación** entre lo esperado y lo obtenido.

**Manual v/s automatizado:**

*   *Manual:* tú ejecutas, miras la salida y juzgas si está bien. Sirve para explorar, pero depende de tu atención.
*   *Automatizado:* las entradas y resultados esperados quedan escritos en código y el computador hace la comparación. Se puede relanzar en segundos cada vez que cambias algo, lo que permite detectar si una modificación **rompió** algo que antes funcionaba.

---

## ¿Qué casos probar?

Una buena batería de pruebas cubre tres zonas del espacio de entradas:

| Tipo | Idea | Ejemplo con `promedio(lista)` |
|---|---|---|
| **Normal** | Un uso típico | `[2, 4, 6]` → `4` |
| **Límite** | Los extremos de lo permitido | `[7]` (un solo elemento) |
| **Inválido** | Lo que el contrato no acepta | `[]` (no existe el promedio de nada) |

---

## `assert`

La instrucción `assert` de Python exige que una condición sea verdadera; si no lo es, se lanza `AssertionError` y el programa se detiene.

```python
def es_par(n):
    return n % 2 == 0

assert es_par(10)
assert not es_par(7)
assert es_par(0)
```

Si no aparece ningún mensaje, todas las condiciones se cumplieron. Sirve para chequeos rápidos, pero tiene dos limitaciones: se detiene en el **primer** fallo (no sabes qué más está roto) y no entrega un reporte ordenado. Para eso existe `unittest`.

---

## `unittest`

Una **prueba unitaria** verifica una pieza pequeña del programa (una función, un método) de forma aislada. Al ser tan acotada, cuando falla sabes exactamente dónde mirar.

`unittest` es built-in de Python.

*   **Test case:** un grupo de pruebas relacionadas, escrito como clase que hereda de `unittest.TestCase`.
*   **Fixture:** el "escenario" que se prepara antes de cada prueba (datos, archivos, etc.) y se limpia después.
*   **Suite:** un conjunto de test cases que se corren juntos.
*   **Runner:** el programa que ejecuta todo y reporta.

Cada método cuyo nombre **empiece con `test`** es una prueba independiente.

```python
import unittest

def promedio(numeros):
    if not numeros:
        raise ValueError("No hay datos")
    return sum(numeros) / len(numeros)

class TestPromedio(unittest.TestCase):
    def test_caso_normal(self):
        self.assertEqual(promedio([2, 4, 6]), 4)

    def test_un_solo_elemento(self):
        self.assertEqual(promedio([7]), 7)

    def test_decimales(self):
        self.assertAlmostEqual(promedio([0.1, 0.2]), 0.15)

    def test_lista_vacia(self):
        with self.assertRaises(ValueError):
            promedio([])
```

**Cómo ejecutarlas:**

```python
# Notebook de Jupyter
unittest.main(argv=[''], exit=False) #este es global, si queremos aislar, agregar parámetro defaultTest='clase test'

# Script .py
if __name__ == "__main__":
    unittest.main()
```

```bash
# Terminal
python -m unittest -v nombre_del_archivo
```

---

## Las comprobaciones disponibles

| Método | Verifica |
|---|---|
| `assertEqual(a, b)` / `assertNotEqual(a, b)` | igualdad / desigualdad |
| `assertTrue(x)` / `assertFalse(x)` | valor de verdad |
| `assertIsNone(x)` | que sea `None` |
| `assertIn(a, b)` | que `a` pertenezca a `b` |
| `assertGreater(a, b)` / `assertLess(a, b)` | comparaciones de orden |
| `assertAlmostEqual(a, b)` | igualdad aproximada (para decimales) |
| `assertCountEqual(a, b)` | mismos elementos, sin importar el orden |
| `assertRaises(Error)` | que se lance cierta excepción |

---

## Evitar repetir código en los tests

*   **`setUp()`:** se ejecuta antes de **cada** test; ideal para armar los datos de partida.
*   **`subTest()`:** corre varios casos dentro de un mismo método y, si alguno falla, indica cuál (sin esto, el primer fallo oculta al resto).

```python
def es_palindromo(texto):
    limpio = texto.lower().replace(" ", "")
    return limpio == limpio[::-1]

class TestPalindromo(unittest.TestCase):
    def setUp(self):
        self.casos = [
            ("reconocer", True),
            ("Anita lava la tina", True),
            ("python", False),
            ("", True),
            ("a", True),
        ]

    def test_varios_textos(self):
        for texto, esperado in self.casos:
            with self.subTest(texto=texto):
                self.assertEqual(es_palindromo(texto), esperado)
```

---

## Leer el resultado de una ejecución

*   **OK:** todo lo ejecutado pasó.
*   **FAIL:** una comprobación dio un resultado distinto del esperado (el código corrió, pero la respuesta es incorrecta).
*   **ERROR:** el código tuvo una excepción que **no** era esperada (por ejemplo, un `TypeError` o `IndexError` dentro de la función).


In [ ]:
#ejecutar primero
import unittest
from collections import deque

# Ejercicios

1. La siguiente función recibe una lista de números y retorna el mayor. Si la lista está vacía, debe lanzar `ValueError`.

In [ ]:
def maximo(numeros):
    mayor = 0
    for n in numeros:
        if n > mayor:
            mayor = n
    return mayor

- Sin leer la implementación con detalle, escribe una clase `TestMaximo` con **al menos 6 tests** que cubran casos normales, límites e inválidos.
- Corrige la función hasta que todos los tests pasen.

In [ ]:
class TestMaximo(unittest.TestCase):
  pass

unittest.main(argv=[''], defaultTest='TestMaximo', exit=False)

2. La función `clasificar_nota(nota)` recibe una nota de la escala chilena y retorna una categoría:
- menor que 4.0	"Reprobado"
- desde 4.0 hasta menos de 5.0	"Suficiente"
- desde 5.0 hasta menos de 6.0	"Bueno"
- desde 6.0 hasta 7.0	"Sobresaliente"+

Considere que:
- La escala válida va de 1.0 a 7.0, ambos incluidos. Fuera de ese rango, lanza `ValueError`.
- Si nota no es int ni float (por ejemplo un str o None), lanza `TypeError`. Booleanos tampoco se consideran notas.

In [ ]:
def clasificar_nota(nota):
    raise NotImplementedError

Para implementar la función, primero debe crear la clase `TestClasificarNota`, la cual debe cubrir:
- al menos un caso normal por categoría,
- todos los valores frontera (justo antes y justo en cada umbral, y los extremos 1.0 y 7.0),
- notas fuera de rango (por encima y por debajo),
- tipos inválidos.

Luego ejecute los test (deben fallar todos, porque aún no hay implementación). Ya después, implemente `clasificar_nota` y ejecute hasta que todos pasen.
Por último, use subTest para probar varias notas dentro de un mismo método sin perder información cuando una falla.

In [ ]:
class TestClasificarNota(unittest.TestCase):
  pass

unittest.main(argv=[''], defaultTest='TestClasificarNota', exit=False)

3. En este ejercicio se le pide implementar el siguiente algoritmo para recorrer grafos:
- **Breadth First Search (BFS):** recorrido en anchura.

Utilice la lista de adyacencia para probar la función. El primer parámetro indica el grafo a recorrer, mientras que el segundo el nodo inicial.

**Tener en cuenta:**
- La función retorna una lista con los nodos en el orden en que se visitan.
- Los vecinos de cada nodo se recorren en el orden en que aparecen en su lista de adyacencia.
- Solo se visitan los nodos alcanzables desde el nodo inicial, y cada uno una sola vez (los ciclos no deben provocar repeticiones ni bucles infinitos).
- Si el nodo inicial no está en el grafo, lanza `ValueError`.

---

### BFS: cómo funciona

BFS explora el grafo por **niveles**: primero el nodo inicial, luego todos sus vecinos, luego los vecinos de esos vecinos, y así sucesivamente. Para lograrlo se usa una **cola**.

1. Verifique que el nodo inicial esté en el grafo.
2. Cree un set `visitados` con el nodo inicial, una cola que contenga el nodo inicial y una lista `orden` vacía.
3. Mientras la cola no esté vacía:
   - Saque el nodo del frente de la cola y agréguelo a `orden`.
   - Para cada vecino de ese nodo que **no** esté en `visitados`: márquelo como visitado y agréguelo al final de la cola.
4. Retorne `orden`.

> Marque cada nodo como visitado **en el momento de agregarlo a la cola**, no al sacarlo. Así nunca queda dos veces en la cola.

**Sobre `deque`**: Una lista de Python no es eficiente como cola: `lista.pop(0)` obliga a desplazar todos los elementos. El módulo `collections` ofrece `deque`, pensado para esto:
- `cola = deque([inicio])`: crea una cola con un elemento.
- `cola.append(x)`: agrega `x` al final.
- `cola.appendleft(x)`: agrega `x` al inicio.
- `cola.popleft()`: saca y retorna el elemento del frente.
- `while cola:`: es falso cuando la cola está vacía.

In [ ]:
grafo = {
    "A": ["B", "C"],
    "B": ["A", "D"],
    "C": ["A", "D"],
    "D": ["B", "C"],
    "E": [], #aislado
}

def bfs(grafo, inicio):
  pass

**Tests**

Una vez implementada, escriba una clase de pruebas con `unittest` para validar `bfs`. No se necesitan muchos tests, pero sí bien elegidos. Escriba el resultado esperado **a partir de la especificación (nota de "tener en cuenta")**, no de lo que devuelve su función. Debe cubrir, como mínimo:

1. **Caso normal:** el orden exacto de BFS desde `"A"` en el grafo de ejemplo.
2. **Nodo aislado:** recorrer desde un nodo sin vecinos.
3. **Ciclo:** un grafo con ciclos (por ejemplo, un triángulo) no debe producir nodos repetidos ni bucles infinitos.
4. **Nodos no alcanzables:** en un grafo con dos componentes separadas, solo debe aparecer la del nodo inicial.
5. **Entrada inválida:** un nodo inicial que no está en el grafo.

In [ ]:
class TestRecorridos(unittest.TestCase):
    def setUp(self):
        self.grafo = {
            "A": ["B", "C"],
            "B": ["A", "D"],
            "C": ["A", "D"],
            "D": ["B", "C"],
            "E": [],
        }

    def test_bfs_orden(self):
        #ejemplo test normal
        self.assertEqual(bfs(self.grafo, "A"), ["A", "B", "C", "D"])

    def test_nodo_aislado(self):
        #completar -> Nodo aislado (recorrer desde "E")
        self.fail("Completar este test")

    def test_ciclo_sin_repetidos(self):
        #completar -> Ciclo: definir un grafo con un ciclo
        self.fail("Completar este test")

    def test_nodos_no_alcanzables(self):
        #completar -> Nodos no alcanzables: definir un grafo con componentes separados
        self.fail("Completar este test")

    def test_inicio_inexistente(self):
        #completar -> nodo inicial que no está en el grafo (ej. probar que lance ValueError)
        self.fail("Completar este test")

unittest.main(argv=[''], defaultTest='TestRecorridos', exit=False)